# Phase 2: Local math assistant with MCP tools

This notebook connects a local Ollama model to a local MCP calculator. Run cells from top to bottom. The model interprets your request; the MCP server performs calculations.

In [ ]:
%%writefile math_server.py
import ast
import math
import operator
from mcp.server.fastmcp import FastMCP

server = FastMCP("local-math")

@server.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@server.tool()
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

@server.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@server.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b

@server.tool()
def square_root(number: float) -> float:
    """Find the square root of a non-negative number."""
    if number < 0:
        raise ValueError("Cannot calculate the square root of a negative number.")
    return math.sqrt(number)

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.UAdd: operator.pos, ast.USub: operator.neg}

def _calculate(node):
    if isinstance(node, ast.Expression):
        return _calculate(node.body)
    if isinstance(node, ast.Constant) and not isinstance(node.value, bool) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp):
        fn = _OPS.get(type(node.op))
        if fn is None:
            raise ValueError("Unsupported operator.")
        left, right = _calculate(node.left), _calculate(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 12:
            raise ValueError("Power must be between -12 and 12.")
        value = fn(left, right)
        if isinstance(value, complex) or not math.isfinite(value) or abs(value) > 1e100:
            raise ValueError("Result is outside the supported range.")
        return value
    if isinstance(node, ast.UnaryOp):
        fn = _OPS.get(type(node.op))
        if fn is not None:
            return fn(_calculate(node.operand))
    raise ValueError("Use only numbers, parentheses, and + - * / % ** operators.")

@server.tool()
def calculate_expression(expression: str) -> str:
    """Calculate a BODMAS arithmetic expression. Supports numbers, parentheses, +, -, *, /, %, **, and ^ for powers."""
    if len(expression) > 200:
        raise ValueError("Expression is too long.")
    expression = expression.replace("^", "**")
    tree = ast.parse(expression, mode="eval")
    if sum(1 for _ in ast.walk(tree)) > 50:
        raise ValueError("Expression is too complex.")
    return f"{expression} = {_calculate(tree)}"

if __name__ == "__main__":
    server.run()


## Configure the local model and MCP server

The important detail is that the MCP server is launched with the notebook's Python interpreter and the exact file created above.

In [ ]:
import sys
from pathlib import Path
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_ollama import ChatOllama
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MODEL_NAME = "qwen3.5:2b"  # Must already be available in local Ollama.
SERVER_FILE = Path("math_server.py").resolve()
if not SERVER_FILE.exists():
    raise FileNotFoundError(f"Server file not found: {SERVER_FILE}")
model = ChatOllama(model=MODEL_NAME, temperature=0)
server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_FILE)])
print("Model:", MODEL_NAME)
print("Server:", SERVER_FILE)
print("Python:", sys.executable)


## Start chatting

This loop calls the agent for every non-empty question and prints its final response. In the original pasted code, the loop stopped after reading input; it never called the agent, which is why no calculation appeared. The MCP server stays open while you chat, avoiding a restart for every question.

In [ ]:
async def math_chat():
    async with stdio_client(server_params, errlog=sys.stderr) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await load_mcp_tools(session)
            print("MCP tools:", [tool.name for tool in tools])
            agent = create_agent(
                model=model,
                tools=tools,
                system_prompt=(
                    "You are a careful math assistant. You MUST call a provided calculator tool for every calculation. "
                    "For expressions with multiple operations, call calculate_expression with only the expression; it follows BODMAS. "
                    "For one operation, call its matching tool. Explain the result briefly and never guess."
                ),
            )
            while True:
                try:
                    question = input("You (type quit to stop): ").strip()
                except (EOFError, KeyboardInterrupt):
                    print("Chat ended.")
                    break
                if question.lower() in {"quit", "exit"}:
                    print("Chat ended.")
                    break
                if not question:
                    continue
                try:
                    result = await agent.ainvoke({"messages": [HumanMessage(content=question)]})
                    messages = result.get("messages", [])
                    if messages:
                        print("Assistant:", messages[-1].content)
                    else:
                        print("Assistant returned no messages.")
                except Exception as error:
                    print(f"Assistant error: {error}")

await math_chat()


## Try these prompts

- What is 3.14 multiplied by 5?
- Add 27 and 35
- What is (12 + 3) * 4 / 2?
- Calculate 2^5 + 10

Type quit to exit.